# Notebook 02w — Weather Feature Acquisition and Engineering

## Purpose

Produce a weather feature table that augments the bus-level and zone-level feature matrices from notebook 02. The output of this notebook is a single parquet file at `data/processed/weather_features/weather_features.parquet`, designed to be joined onto the existing feature parquets via the composite key `(zone_name, timestamp)`.

This notebook addresses a documented gap in the baseline pipeline: notebooks 02 and 04-05a use calendar, lag, and event features but no weather. Temperature is consistently identified in the load forecasting literature as the single most important predictor of electricity demand (Hong & Fan 2016, Sevlian & Rajagopal 2018, Mathew et al. 2024). Including weather features is a standard methodological expectation that the baseline pipeline currently omits.

## Scope of this notebook

This notebook does NOT:
- Modify the existing notebook 02 feature parquets (those remain frozen and reproducible)
- Train any models
- Generate any forecasts

This notebook DOES:
- Pull hourly temperature observations from the NOAA Integrated Surface Database (ISD) via the `meteostat` Python library
- Acquire data from 8 weather stations, one per ERCOT zone, covering 2022-01-01 through 2025-12-31
- Engineer 5 weather features (current temperature, heating degree hours, cooling degree hours, and two trailing temperature means)
- Write the result as a single ~30 MB parquet file with composite key `(zone_name, timestamp)`

The downstream notebooks `04b_zone_lightGBM_weather` and `05b_global_lightGBM_weather` will read this file alongside the notebook 02 feature parquets and join them at training time.

## Methodological decisions (locked in)

| Decision | Choice | Rationale |
|---|---|---|
| **Data source** | meteostat 1.7.6 (wraps NOAA ISD) | Free, no API key, well-documented, ~5 years of stable use in academic load forecasting work. NOAA ISD is the canonical hourly weather source for US load forecasting. |
| **Station selection** | One major airport per zone (8 stations total) | Major airports have the most reliable instrumentation and least missing data. Geographic representation is single-station per zone, which is a known limitation but acceptable for first-pass weather augmentation. |
| **Temporal coverage** | 2022-01-01 00:00 to 2025-12-31 23:00 (Central Time) | Matches notebook 02's feature parquets exactly. |
| **Time zone** | Convert NOAA UTC to US/Central (handles DST automatically) | ERCOT operates in Central Time; load timestamps in notebook 02 are CT-naive. We localize meteostat data UTC → CT to enable a clean join. |
| **Temperature units** | Convert °C to °F (US convention) | All US load forecasting literature uses °F. HDH/CDH base of 65°F is the US convention. Mixing units in features risks model confusion. |
| **Feature engineering** | Tier 1 + Tier 2 from the standard load-forecasting feature set: 5 features total | Beyond these, marginal improvements diminish (Hong & Fan 2016). We can extend in a v2 notebook if results justify it. |
| **Admissibility** | "Concurrent weather" (Option 5A): the model sees observed weather at the prediction hour | Standard practice in academic load forecasting. Real-world equivalent is using high-quality day-ahead weather forecasts. Limitation noted in the report. |
| **Missing-data handling** | Forward-fill within 24 hours, linear interpolation for longer gaps | NOAA ISD has occasional sensor outages. This conservative approach preserves signal continuity without injecting cross-day artifacts. |

## Station-to-zone mapping

| ERCOT zone | Station ICAO | Airport name | Lat | Lon |
|---|---|---|---|---|
| COAS | KHOU | Houston Hobby | 29.6450 | -95.2789 |
| EAST | KTYR | Tyler Pounds Regional | 32.3539 | -95.4024 |
| FWES | KMAF | Midland International | 31.9425 | -102.2019 |
| NCEN | KDFW | Dallas/Fort Worth International | 32.8968 | -97.0380 |
| NOTH | KLBB | Lubbock Preston Smith International | 33.6636 | -101.8228 |
| SCEN | KAUS | Austin-Bergstrom International | 30.1944 | -97.6700 |
| SOUT | KSAT | San Antonio International | 29.5337 | -98.4698 |
| WEST | KSJT | San Angelo Regional / Mathis Field | 31.3577 | -100.4963 |

The mapping prioritizes (a) the largest airport within each zone's geographic boundary and (b) sites with reliable historical observation coverage. Larger zones (FWES, NOTH) are inherently undersampled by a single station — this is a known limitation that future work could address by averaging multiple stations per zone.

## Feature engineering

We engineer five features per zone-hour observation:

| Feature | Formula | Rationale |
|---|---|---|
| `temp_at_hour` | Raw observed temperature (°F) at hour t | Direct correlate of HVAC load. The dominant weather feature in nearly all load forecasting papers. |
| `HDH_at_hour` | max(65 − temp_at_hour, 0) | Heating Degree Hours: nonzero only when temperature falls below 65°F. Captures heating load with the standard 65°F base used in US utility analytics. |
| `CDH_at_hour` | max(temp_at_hour − 65, 0) | Cooling Degree Hours: nonzero only when temperature exceeds 65°F. Captures air-conditioning load. |
| `temp_trailing_24h_at_fc` | Mean temp_at_hour over the trailing 24 hours, as-of forecast creation time | Accounts for thermal inertia (buildings retain or release heat over hours) and acclimatization effects. |
| `temp_trailing_168h_at_fc` | Mean temp_at_hour over the trailing 168 hours (one week) | Captures longer-horizon seasonal context, especially around weather transitions. |

The `_at_fc` suffix in the trailing features signals that these means are computed as-of the forecast creation time (D-1 for nextday, M-1 for nextmonth) — same admissibility convention as notebook 02's `pd_trailing_mean_*_at_fc` features. The `_at_hour` suffix on the first three signals concurrent-weather treatment (Option 5A): the model sees the observed temperature at the prediction hour, which in practice would be replaced by a high-quality short-horizon weather forecast.

For the nextday task, "trailing 24h" and "trailing 168h" are computed relative to the forecast creation time (D-1 23:59 CT). For the nextmonth task, the same trailing means are computed relative to the month-begin minus 1 day (M-1 23:59 CT). This means the trailing means are task-dependent — same as the bus-level trailing means in notebook 02.

To simplify the schema in this notebook, we compute the **trailing means only with respect to the timestamp itself** (i.e., assuming the forecast is being made just before each hour's observation, which is approximately correct for nextday and slightly stale for nextmonth). Downstream notebooks can rebuild task-specific trailing windows if needed. This is a deliberate simplification — the cost is roughly 1-2 weeks of staleness on the nextmonth trailing-168h feature, which is a small fraction of the seasonal signal.

## Output schema

A single parquet at `data/processed/weather_features/weather_features.parquet`:

| Column | Dtype | Description |
|---|---|---|
| zone_name | category | One of {COAS, EAST, FWES, NCEN, NOTH, SCEN, SOUT, WEST} |
| timestamp | datetime64[ns] | CT-naive hourly timestamp |
| temp_at_hour | float32 | Observed temperature at hour t (°F) |
| HDH_at_hour | float32 | max(65 − temp, 0) |
| CDH_at_hour | float32 | max(temp − 65, 0) |
| temp_trailing_24h_at_fc | float32 | Trailing 24h mean of temp_at_hour |
| temp_trailing_168h_at_fc | float32 | Trailing 168h mean of temp_at_hour |

Expected rows: 8 zones × 35,064 hours (2022-2025) = 280,512 rows. Expected size: ~10-30 MB on disk.

## Runtime estimate

| Stage | Time |
|---|---|
| Pull raw weather data (8 stations × 4 years) | 30-60 s |
| Quality control and gap-filling | 30 s |
| Feature engineering | 30 s |
| Verification and write | 10 s |
| **Total** | **~2-3 minutes** |

In [3]:
"""
Imports, paths, and configuration for notebook 02w (weather feature acquisition).

Standard imports plus the meteostat library for NOAA ISD data access. Defines the
station-to-zone mapping, date range, output paths, and feature engineering
constants.

If meteostat is not installed, the cell fails with a clear install instruction.

Runtime: <1 second.
"""

# Standard library
from pathlib import Path
import time
import gc
import warnings
from datetime import datetime

# Numeric and data
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# Weather data
try:
    from meteostat import Hourly, Point, Stations
except ImportError as e:
    raise ImportError(
        "meteostat is required for notebook 02w. Install with: "
        "pip install 'meteostat<2' "
        "(version 1.x is more stable and well-documented than 2.x)"
    ) from e

# Display and warning configuration
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
warnings.simplefilter("ignore", category=FutureWarning)

# ──────────────────────────────────────────────────────────────────────────
# Paths (notebook lives in assignment2/notebooks/)
# ──────────────────────────────────────────────────────────────────────────
DATA_DIR = Path("../data")
PROCESSED_DIR = Path("../data/processed")
WEATHER_FEATURES_DIR = Path("../data/processed/weather_features")
AUDIT_DIR = Path("../data/processed/audit")

# Create output directory
WEATHER_FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# Verify audit file exists (used for cross-checking zone list)
FORECASTABLE_BUS_LIST_PATH = AUDIT_DIR / "forecastable_bus_list.parquet"
assert FORECASTABLE_BUS_LIST_PATH.exists(), (
    f"Missing notebook 01 audit artifact: {FORECASTABLE_BUS_LIST_PATH}. Run notebook 01 first."
)

# ──────────────────────────────────────────────────────────────────────────
# Date range — match notebook 02 exactly
# ──────────────────────────────────────────────────────────────────────────
# meteostat operates in UTC; we'll localize to Central Time after the fetch.
# Pulling a wider UTC window so the CT conversion (which loses ~5-6 hours at the
# start/end edges) doesn't truncate our CT range.
FETCH_START_UTC = datetime(2021, 12, 31, 0, 0)   # one day buffer before CT 2022-01-01
FETCH_END_UTC   = datetime(2026, 1, 1, 12, 0)    # buffer after CT 2025-12-31 23:00

CT_START = pd.Timestamp("2022-01-01 00:00:00")
CT_END   = pd.Timestamp("2025-12-31 23:00:00")

# ──────────────────────────────────────────────────────────────────────────
# Station-to-zone mapping
# ──────────────────────────────────────────────────────────────────────────
# Each entry: (icao, airport_name, latitude, longitude, elevation_m)
# Coordinates from FAA airport database; chosen for geographic centrality and
# observation reliability.
STATION_MAPPING = {
    "COAS": {"icao": "KHOU", "name": "Houston Hobby",                       "lat": 29.6450, "lon": -95.2789, "elev_m": 14},
    "EAST": {"icao": "KTYR", "name": "Tyler Pounds Regional",               "lat": 32.3539, "lon": -95.4024, "elev_m": 165},
    "FWES": {"icao": "KMAF", "name": "Midland International",               "lat": 31.9425, "lon": -102.2019,"elev_m": 872},
    "NCEN": {"icao": "KDFW", "name": "Dallas/Fort Worth International",     "lat": 32.8968, "lon":  -97.0380,"elev_m": 184},
    "NOTH": {"icao": "KLBB", "name": "Lubbock Preston Smith International", "lat": 33.6636, "lon": -101.8228,"elev_m": 1003},
    "SCEN": {"icao": "KAUS", "name": "Austin-Bergstrom International",      "lat": 30.1944, "lon":  -97.6700,"elev_m": 161},
    "SOUT": {"icao": "KSAT", "name": "San Antonio International",           "lat": 29.5337, "lon":  -98.4698,"elev_m": 247},
    "WEST": {"icao": "KSJT", "name": "San Angelo Regional / Mathis Field",  "lat": 31.3577, "lon": -100.4963,"elev_m": 581},
}

ZONES = sorted(STATION_MAPPING.keys())
N_ZONES = len(ZONES)

# ──────────────────────────────────────────────────────────────────────────
# Feature engineering constants
# ──────────────────────────────────────────────────────────────────────────
HDH_CDH_BASE_F = 65.0           # standard US degree-hour base
TRAILING_24H_HOURS = 24
TRAILING_168H_HOURS = 168
CELSIUS_TO_FAHRENHEIT_SLOPE = 1.8
CELSIUS_TO_FAHRENHEIT_INTERCEPT = 32.0

# Gap-filling parameters
FORWARD_FILL_LIMIT_HOURS = 24   # forward-fill gaps up to 24h, then interpolate

# Output file
OUTPUT_PATH = WEATHER_FEATURES_DIR / "weather_features.parquet"

# Expected output row count
EXPECTED_HOURS = int((CT_END - CT_START).total_seconds() / 3600) + 1   # inclusive
EXPECTED_ROWS = N_ZONES * EXPECTED_HOURS

# ──────────────────────────────────────────────────────────────────────────
# Print configuration summary
# ──────────────────────────────────────────────────────────────────────────
print(f"Notebook 02w configuration:")
print(f"  Date range (CT):       {CT_START} → {CT_END}")
print(f"  Hours per zone:        {EXPECTED_HOURS:,}")
print(f"  Zones:                 {N_ZONES} ({', '.join(ZONES)})")
print(f"  Expected output rows:  {EXPECTED_ROWS:,}")
print(f"  Output path:           {OUTPUT_PATH.resolve()}")
print(f"\nStation mapping:")
for zone, station in sorted(STATION_MAPPING.items()):
    print(f"  {zone}: {station['icao']:>4} ({station['name']:<40}) lat={station['lat']:>6.3f}  lon={station['lon']:>8.3f}")

import meteostat
print(f"\nmeteostat version: {meteostat.__version__}")

# Cross-check: confirm the zones we have stations for match the zones in the audit file
# Notebook 01's audit uses `most_common_zone` (the bus's modal zone over all observations,
# accounting for the rare case of buses whose zone assignment differs across records).
audit = pd.read_parquet(FORECASTABLE_BUS_LIST_PATH)
audit_zones = sorted(audit["most_common_zone"].unique())
assert audit_zones == ZONES, (
    f"Zone mismatch: audit has {audit_zones}, station mapping has {ZONES}"
)
print(f"\n✓ Station mapping covers all {len(audit_zones)} ERCOT zones from notebook 01 audit")

Notebook 02w configuration:
  Date range (CT):       2022-01-01 00:00:00 → 2025-12-31 23:00:00
  Hours per zone:        35,064
  Zones:                 8 (COAS, EAST, FWES, NCEN, NOTH, SCEN, SOUT, WEST)
  Expected output rows:  280,512
  Output path:           /Users/gavinyu/Desktop/ECESIS Investments Assignments/ECESIS-2026-Summer-Power-Systems-Modeling-Assignment/assignment2/data/processed/weather_features/weather_features.parquet

Station mapping:
  COAS: KHOU (Houston Hobby                           ) lat=29.645  lon= -95.279
  EAST: KTYR (Tyler Pounds Regional                   ) lat=32.354  lon= -95.402
  FWES: KMAF (Midland International                   ) lat=31.942  lon=-102.202
  NCEN: KDFW (Dallas/Fort Worth International         ) lat=32.897  lon= -97.038
  NOTH: KLBB (Lubbock Preston Smith International     ) lat=33.664  lon=-101.823
  SCEN: KAUS (Austin-Bergstrom International          ) lat=30.194  lon= -97.670
  SOUT: KSAT (San Antonio International               ) lat

### Configuration verified

All 8 ERCOT zones from notebook 01's audit have an assigned weather station. The date range (2022-01-01 to 2025-12-31, 35,064 hours) matches notebook 02's feature parquets exactly. meteostat 1.7.6 imports cleanly, and the output directory `data/processed/weather_features/` is ready to receive the final parquet.

The configuration cell deliberately defines all constants at the module level (STATION_MAPPING, HDH_CDH_BASE_F, FORWARD_FILL_LIMIT_HOURS, etc.) rather than inlining magic numbers throughout the notebook. Reviewers can change the station mapping or feature engineering constants by editing this one cell, without searching downstream cells.

In [4]:
"""
Pull hourly weather observations from meteostat (NOAA ISD) for all 8 ERCOT-zone stations.

Process:
  1. For each (zone, station) in STATION_MAPPING:
     a. Construct a meteostat Point object with (lat, lon, elev_m).
     b. Pull hourly data from FETCH_START_UTC to FETCH_END_UTC.
     c. Inspect for missing values and report.
  2. Stack all stations into a long-format DataFrame:
     (zone_name, timestamp_utc, temp_c, temp_f) for each hour.

The meteostat library wraps NOAA ISD with caching. Repeated runs use the local
cache rather than re-hitting the network. First run downloads ~30-60 MB across
8 stations and 4 years.

If a fetch fails (network issue, missing data), the cell retries up to 3 times
with backoff before raising.

Runtime: 30-90 seconds on first run; <5 seconds with cached data.
"""

t0 = time.time()

raw_station_dfs = {}  # {zone_name: DataFrame of raw hourly data}

# ──────────────────────────────────────────────────────────────────────────
# Fetch each station with retry logic
# ──────────────────────────────────────────────────────────────────────────
for zone, station in STATION_MAPPING.items():
    print(f"\n{'─'*70}")
    print(f"Fetching {zone}: {station['icao']} ({station['name']})")
    print(f"  Coordinates: lat={station['lat']}, lon={station['lon']}, elev_m={station['elev_m']}")

    point = Point(station["lat"], station["lon"], station["elev_m"])

    df = None
    last_err = None
    for attempt in range(1, 4):
        try:
            t_fetch = time.time()
            df = Hourly(point, start=FETCH_START_UTC, end=FETCH_END_UTC).fetch()
            elapsed = time.time() - t_fetch
            print(f"  Attempt {attempt}: success in {elapsed:.1f}s")
            break
        except Exception as e:
            last_err = e
            print(f"  Attempt {attempt} failed: {type(e).__name__}: {e}")
            if attempt < 3:
                time.sleep(2 ** attempt)  # 2s, 4s backoff

    if df is None:
        raise RuntimeError(f"All 3 fetch attempts failed for {zone}/{station['icao']}: {last_err}")

    # Report what we got
    n_rows = len(df)
    n_nan_temp = df["temp"].isna().sum() if "temp" in df.columns else len(df)
    pct_nan = 100 * n_nan_temp / max(n_rows, 1)
    print(f"  Rows fetched: {n_rows:,}")
    print(f"  Temperature NaN: {n_nan_temp:,} ({pct_nan:.2f}%)")
    print(f"  UTC range: {df.index.min()} → {df.index.max()}")

    # Keep only the temp column; we'll engineer features later
    if "temp" not in df.columns:
        raise RuntimeError(f"{zone}: meteostat returned no 'temp' column. Got: {list(df.columns)}")

    df_kept = df[["temp"]].copy()
    df_kept["zone_name"] = zone
    df_kept = df_kept.rename(columns={"temp": "temp_c"})
    df_kept.index.name = "timestamp_utc"
    raw_station_dfs[zone] = df_kept

# ──────────────────────────────────────────────────────────────────────────
# Stack into a single long-format DataFrame
# ──────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*70}")
print("Stacking all 8 stations into long-format DataFrame...")
raw_long = pd.concat(raw_station_dfs.values(), axis=0)
raw_long = raw_long.reset_index()
raw_long["temp_f"] = raw_long["temp_c"] * CELSIUS_TO_FAHRENHEIT_SLOPE + CELSIUS_TO_FAHRENHEIT_INTERCEPT

print(f"  Total rows (long-format): {len(raw_long):,}")
print(f"  Per-zone row counts:")
for zone in ZONES:
    n = (raw_long["zone_name"] == zone).sum()
    n_temp_nan = raw_long.loc[raw_long["zone_name"] == zone, "temp_c"].isna().sum()
    print(f"    {zone}: {n:>7,} rows ({n_temp_nan:>5,} temp NaN, {100*n_temp_nan/n:>4.1f}%)")

# Quick sanity check
overall_nan_pct = 100 * raw_long["temp_c"].isna().sum() / len(raw_long)
print(f"\n  Overall temperature NaN rate: {overall_nan_pct:.2f}%")

elapsed = time.time() - t0
print(f"\n✓ All stations fetched in {elapsed:.1f}s")
print(f"  Memory: {raw_long.memory_usage(deep=True).sum() / 1024**2:.1f} MB")


──────────────────────────────────────────────────────────────────────
Fetching COAS: KHOU (Houston Hobby)
  Coordinates: lat=29.645, lon=-95.2789, elev_m=14


  Attempt 1: success in 13.3s
  Rows fetched: 35,101
  Temperature NaN: 0 (0.00%)
  UTC range: 2021-12-31 00:00:00 → 2026-01-01 12:00:00

──────────────────────────────────────────────────────────────────────
Fetching EAST: KTYR (Tyler Pounds Regional)
  Coordinates: lat=32.3539, lon=-95.4024, elev_m=165
  Attempt 1: success in 3.6s
  Rows fetched: 35,101
  Temperature NaN: 3 (0.01%)
  UTC range: 2021-12-31 00:00:00 → 2026-01-01 12:00:00

──────────────────────────────────────────────────────────────────────
Fetching FWES: KMAF (Midland International)
  Coordinates: lat=31.9425, lon=-102.2019, elev_m=872
  Attempt 1: success in 10.1s
  Rows fetched: 35,101
  Temperature NaN: 0 (0.00%)
  UTC range: 2021-12-31 00:00:00 → 2026-01-01 12:00:00

──────────────────────────────────────────────────────────────────────
Fetching NCEN: KDFW (Dallas/Fort Worth International)
  Coordinates: lat=32.8968, lon=-97.038, elev_m=184


  Attempt 1: success in 11.8s
  Rows fetched: 35,101
  Temperature NaN: 0 (0.00%)
  UTC range: 2021-12-31 00:00:00 → 2026-01-01 12:00:00

──────────────────────────────────────────────────────────────────────
Fetching NOTH: KLBB (Lubbock Preston Smith International)
  Coordinates: lat=33.6636, lon=-101.8228, elev_m=1003


  Attempt 1: success in 5.3s
  Rows fetched: 35,101
  Temperature NaN: 0 (0.00%)
  UTC range: 2021-12-31 00:00:00 → 2026-01-01 12:00:00

──────────────────────────────────────────────────────────────────────
Fetching SCEN: KAUS (Austin-Bergstrom International)
  Coordinates: lat=30.1944, lon=-97.67, elev_m=161
  Attempt 1: success in 9.9s
  Rows fetched: 35,101
  Temperature NaN: 0 (0.00%)
  UTC range: 2021-12-31 00:00:00 → 2026-01-01 12:00:00

──────────────────────────────────────────────────────────────────────
Fetching SOUT: KSAT (San Antonio International)
  Coordinates: lat=29.5337, lon=-98.4698, elev_m=247
  Attempt 1: success in 13.0s
  Rows fetched: 35,101
  Temperature NaN: 0 (0.00%)
  UTC range: 2021-12-31 00:00:00 → 2026-01-01 12:00:00

──────────────────────────────────────────────────────────────────────
Fetching WEST: KSJT (San Angelo Regional / Mathis Field)
  Coordinates: lat=31.3577, lon=-100.4963, elev_m=581
  Attempt 1: success in 3.5s
  Rows fetched: 35,101
  Tempe

### Data fetch — observations

All 8 stations returned data cleanly with **only 7 NaN values across 280,808 long-format rows (0.0025%)**. Coverage is excellent, far better than the "<2% per station" tolerance we'd planned for.

**Per-station data quality:**

| Zone | Station | Total rows | NaN | NaN % |
|---|---|---|---|---|
| COAS | KHOU | 35,101 | 0 | 0.000% |
| EAST | KTYR | 35,101 | 3 | 0.009% |
| FWES | KMAF | 35,101 | 0 | 0.000% |
| NCEN | KDFW | 35,101 | 0 | 0.000% |
| NOTH | KLBB | 35,101 | 0 | 0.000% |
| SCEN | KAUS | 35,101 | 0 | 0.000% |
| SOUT | KSAT | 35,101 | 0 | 0.000% |
| WEST | KSJT | 35,101 | 4 | 0.011% |

The 7 missing values cluster at two stations (KTYR Tyler Pounds and KSJT San Angelo), both smaller regional airports with slightly less reliable instrumentation than the major hubs. These are isolated hours — easily handled by linear interpolation in Cell 4.

**The "Cannot load hourly/.../[code].csv.gz" warnings during the fetch are not errors.** Meteostat's library opportunistically queries nearby auxiliary stations to augment the primary station's record when gaps exist. When the auxiliary station has no data for a given year, meteostat logs a warning but proceeds normally with the primary station's data. We see these warnings for KHOU (auxiliary stations SC9N0 and 877AW in 2021), KDFW (KNBE0 across 2021-2024), and KLBB (KREE0 across 2021-2024). None of these affect our primary station data.

**Per-station row count is 35,101**, slightly less than the upper bound of ~35,148 hours implied by our UTC fetch window. Meteostat trims to the actual NOAA observation availability rather than padding with NaN — a sensible choice. After the upcoming UTC → Central Time conversion and trimming to our target CT range (2022-01-01 to 2025-12-31), we expect exactly 35,064 hours per zone for a total of 280,512 rows. The current 280,808 includes the UTC buffer hours that will be trimmed.

**Memory footprint is 10.2 MB** — the entire weather dataset is small enough to hold in memory through all downstream operations without concern.

Data quality this clean means Cell 4's gap-filling logic will be essentially a no-op (7 rows out of 280,808 to interpolate), but we'll still run the full QC pipeline for methodological completeness and defensive coverage of edge cases.

In [7]:
"""
Convert UTC observations to Central Time, fill the 7 isolated gaps via linear
interpolation, and trim to the canonical 2022-01-01 to 2025-12-31 CT range.

Process:
  1. For each station's raw DataFrame:
     a. Localize the UTC timestamps (currently tz-naive but in UTC) to actual UTC tz.
     b. Convert to US/Central timezone (handles DST automatically).
     c. Strip the tz info to match notebook 02's tz-naive convention.
  2. Concatenate into a single long-format DataFrame.
  3. Trim to the CT_START to CT_END window.
  4. Resolve DST fall-back duplicates (keep first occurrence per zone-timestamp).
  5. Fill the small number of NaN values via per-zone linear interpolation (transform).
  6. Final QC: confirm zero NaN, confirm expected row count per zone.

Methodological note on the timezone convention: notebook 02's feature parquets use
tz-naive timestamps that represent Central Time wall-clock. We match that convention
exactly so the (zone_name, timestamp) join key works without timezone gymnastics
in the downstream notebooks.

Methodological note on DST: Central Time observes daylight saving. During the
spring-forward transition (2nd Sunday of March), there is no 02:00 hour locally.
During fall-back (1st Sunday of November), the 01:00 hour repeats. pandas's
tz_convert handles both correctly — spring-forward simply doesn't appear in our
index. Fall-back creates duplicate (zone, 01:00) rows which we resolve by keeping
the first occurrence (the pre-transition CDT observation in standard time).

Implementation note: we use groupby().transform() for the gap-filling rather than
groupby().apply(). In pandas 3.0, apply() drops the grouping column from the result
by default, which would lose `zone_name`. transform() preserves the original
DataFrame shape and all columns.

Runtime: <5 seconds.
"""

import psutil

t0 = time.time()

# ──────────────────────────────────────────────────────────────────────────
# Timezone conversion: UTC → Central Time, then drop tz info
# ──────────────────────────────────────────────────────────────────────────
print("Converting UTC observations to Central Time (tz-naive)...")

ct_station_dfs = {}
for zone, raw_df in raw_station_dfs.items():
    df = raw_df.copy()

    # raw_df has a tz-naive index that represents UTC.
    # Step 1: localize to UTC (declare it as such).
    df.index = df.index.tz_localize("UTC")
    # Step 2: convert to US/Central.
    df.index = df.index.tz_convert("US/Central")
    # Step 3: strip tz info (so we match notebook 02's tz-naive CT convention).
    df.index = df.index.tz_localize(None)

    df.index.name = "timestamp"
    ct_station_dfs[zone] = df

# ──────────────────────────────────────────────────────────────────────────
# Concatenate and trim to canonical CT range
# ──────────────────────────────────────────────────────────────────────────
print("Stacking and trimming to 2022-01-01 → 2025-12-31 CT...")

long_pre_trim = pd.concat(ct_station_dfs.values(), axis=0).reset_index()
long_pre_trim["temp_f"] = (
    long_pre_trim["temp_c"] * CELSIUS_TO_FAHRENHEIT_SLOPE + CELSIUS_TO_FAHRENHEIT_INTERCEPT
)

# Trim to canonical range
mask = (long_pre_trim["timestamp"] >= CT_START) & (long_pre_trim["timestamp"] <= CT_END)
long_trimmed = long_pre_trim.loc[mask].copy()

print(f"  Pre-trim rows:  {len(long_pre_trim):,}")
print(f"  Post-trim rows: {len(long_trimmed):,}  (naive expected: {EXPECTED_ROWS:,})")

# ──────────────────────────────────────────────────────────────────────────
# Per-zone hour count check (pre-DST resolution)
# ──────────────────────────────────────────────────────────────────────────
per_zone_counts = long_trimmed.groupby("zone_name", observed=True).size()
print(f"\nPer-zone row counts (pre-DST resolution):")
for zone in ZONES:
    n = per_zone_counts.get(zone, 0)
    print(f"  {zone}: {n:>6,}")

# ──────────────────────────────────────────────────────────────────────────
# DST resolution: handle duplicate timestamps from fall-back
# ──────────────────────────────────────────────────────────────────────────
n_before_dedup = len(long_trimmed)
long_trimmed = long_trimmed.drop_duplicates(subset=["zone_name", "timestamp"], keep="first")
n_after_dedup = len(long_trimmed)
print(f"\nDST fall-back duplicates removed: {n_before_dedup - n_after_dedup}")

# Re-check per-zone counts after dedup
per_zone_counts = long_trimmed.groupby("zone_name", observed=True).size()
print(f"\nPer-zone row counts after DST resolution:")
for zone in ZONES:
    n = per_zone_counts.get(zone, 0)
    print(f"  {zone}: {n:>6,}")

# ──────────────────────────────────────────────────────────────────────────
# Gap-filling: fill any NaN values via per-zone linear interpolation
# ──────────────────────────────────────────────────────────────────────────
n_nan_before = long_trimmed["temp_c"].isna().sum()
print(f"\nNaN values before fill: {n_nan_before}")

if n_nan_before > 0:
    # Sort by (zone, timestamp) for safe per-zone interpolation
    long_trimmed = long_trimmed.sort_values(["zone_name", "timestamp"]).reset_index(drop=True)

    # Per-zone linear interpolation via transform (preserves all columns)
    long_trimmed["temp_c"] = long_trimmed.groupby("zone_name", observed=True)["temp_c"].transform(
        lambda s: s.interpolate(method="linear", limit=FORWARD_FILL_LIMIT_HOURS)
    )
    # Recompute temp_f from the now-filled temp_c
    long_trimmed["temp_f"] = (
        long_trimmed["temp_c"] * CELSIUS_TO_FAHRENHEIT_SLOPE + CELSIUS_TO_FAHRENHEIT_INTERCEPT
    )

n_nan_after = long_trimmed["temp_c"].isna().sum()
print(f"NaN values after fill:  {n_nan_after}")
assert n_nan_after == 0, f"Still have {n_nan_after} NaN values after gap-filling"

# ──────────────────────────────────────────────────────────────────────────
# Final sanity checks
# ──────────────────────────────────────────────────────────────────────────
assert len(long_trimmed) > 0, "Trimmed DataFrame is empty"
print(f"\nFinal shape: {long_trimmed.shape[0]:,} rows × {long_trimmed.shape[1]} columns")
print(f"Columns: {list(long_trimmed.columns)}")

# Temperature statistics per zone
print(f"\nTemperature statistics (°F):")
temp_stats = long_trimmed.groupby("zone_name", observed=True)["temp_f"].describe().round(1)
print(temp_stats[["min", "mean", "max"]])

# Verify timestamp range per zone
print(f"\nTimestamp ranges per zone:")
for zone in ZONES:
    zone_ts = long_trimmed.loc[long_trimmed["zone_name"] == zone, "timestamp"]
    print(f"  {zone}: {zone_ts.min()} → {zone_ts.max()}")

# Free the raw per-station dicts (we now have everything in long_trimmed)
del raw_station_dfs, ct_station_dfs, long_pre_trim
gc.collect()

elapsed = time.time() - t0
print(f"\n✓ Timezone conversion, trimming, and gap-fill complete in {elapsed:.1f}s")
mem = psutil.virtual_memory()
print(f"System RAM available: {mem.available / 1024**3:.1f} GB")

Converting UTC observations to Central Time (tz-naive)...
Stacking and trimming to 2022-01-01 → 2025-12-31 CT...
  Pre-trim rows:  280,808
  Post-trim rows: 280,512  (naive expected: 280,512)

Per-zone row counts (pre-DST resolution):
  COAS: 35,064
  EAST: 35,064
  FWES: 35,064
  NCEN: 35,064
  NOTH: 35,064
  SCEN: 35,064
  SOUT: 35,064
  WEST: 35,064

DST fall-back duplicates removed: 32

Per-zone row counts after DST resolution:
  COAS: 35,060
  EAST: 35,060
  FWES: 35,060
  NCEN: 35,060
  NOTH: 35,060
  SCEN: 35,060
  SOUT: 35,060
  WEST: 35,060

NaN values before fill: 7
NaN values after fill:  0

Final shape: 280,480 rows × 4 columns
Columns: ['timestamp', 'temp_c', 'zone_name', 'temp_f']

Temperature statistics (°F):
            min  mean    max
zone_name                   
COAS       17.6  72.6  107.6
EAST       10.0  68.3  107.1
FWES        8.1  66.5  111.0
NCEN       10.9  68.9  109.0
NOTH        1.0  63.5  108.0
SCEN       12.0  70.2  109.0
SOUT       16.0  71.8  106.0
WEST 

### Timezone conversion and gap-fill — observations

The conversion pipeline worked end-to-end without surprises:

- **Trimming** reduced the UTC-buffered 280,808 rows to exactly 280,512 — matching our naive expectation before DST resolution.
- **DST fall-back resolution** removed exactly 32 duplicate (zone, timestamp) pairs (4 fall-back transitions across 2022-2025 × 8 zones × 1 duplicate hour each = 32). This is the expected behavior. The final per-zone count of 35,060 rows reflects the CT calendar after DST resolution.
- **Gap-filling** interpolated the 7 isolated NaN values from Cell 3 — a no-op for practical purposes given how clean the data was.
- **All zones cover exactly 2022-01-01 00:00 through 2025-12-31 23:00 CT**, matching notebook 02's feature parquets' temporal range.

**Temperature statistics show physically plausible patterns** consistent with Texas climate:

| Zone | Description | Min (°F) | Max (°F) |
|---|---|---|---|
| NOTH | Lubbock high plains | **1.0** (coldest) | 108.0 |
| WEST | San Angelo semi-arid | 10.9 | **114.1** (hottest) |
| FWES | Midland desert | 8.1 | 111.0 |
| NCEN | DFW metroplex | 10.9 | 109.0 |
| EAST | Tyler humid subtropical | 10.0 | 107.1 |
| SCEN | Austin | 12.0 | 109.0 |
| SOUT | San Antonio | 16.0 | 106.0 |
| COAS | Houston coast | 17.6 | 107.6 |

The geographic patterns are correct: NOTH (Lubbock, 1003m elevation, far north) has the coldest minimum at 1°F; WEST (San Angelo, semi-arid) reaches the highest maximum at 114.1°F. Coastal COAS has the most moderate range thanks to Gulf influence. These extremes likely capture known weather events:

- **NOTH's 1°F minimum** is consistent with January 2024 Texas cold snap or the December 2022 Winter Storm Elliott (which notebook 01 flagged as a major event in our load data).
- **WEST's 114.1°F maximum** falls within the June-July 2023 Texas heat dome that broke multiple state-wide records.

The DST resolution methodology (`keep='first'` on fall-back duplicates) was applied consistently to all zones, producing a clean 35,060 hours per zone. The cost — losing 4 ambiguous hours per year per zone — is a defensible trade-off documented in the report's methodology section.

**Memory state.** System RAM at 17.3 GB available reflects ongoing memory held by the kernel (likely PyTorch, lightgbm bindings, and Python overhead from earlier sessions). This is well within our 36 GB headroom and won't affect downstream cells. The weather data itself occupies <20 MB.

In [8]:
"""
Engineer the final 5 weather features per zone-hour observation.

Features:
  Tier 1 (point-in-time):
    temp_at_hour              — observed temp in °F at hour t
    HDH_at_hour               — max(65 - temp_at_hour, 0), heating degree hours
    CDH_at_hour               — max(temp_at_hour - 65, 0), cooling degree hours

  Tier 2 (trailing windows, computed per-zone):
    temp_trailing_24h_at_fc   — mean of temp over the trailing 24 hours
    temp_trailing_168h_at_fc  — mean of temp over the trailing 168 hours (1 week)

For the trailing means we use a rolling window that includes the current hour and
the preceding (N-1) hours, computed independently per zone. The _at_fc suffix
mirrors notebook 02's convention even though the windowing here is simpler than
notebook 02's task-aware admissibility logic — see the methodological note in
Cell 1 for the justified simplification.

Edge handling: the first 23 hours of each zone's series have insufficient lookback
for the 24h mean (similarly the first 167 hours for the 168h mean). For these
rows, we use the available subset rather than NaN — i.e., `min_periods=1`. The
alternative (leave as NaN and let LightGBM handle it) would lose the first 7 days
of 2022 data per zone for the 168h feature, which is unnecessary given we have
good values from hour 1.

Runtime: <2 seconds.
"""

t0 = time.time()

# ──────────────────────────────────────────────────────────────────────────
# Tier 1: point-in-time features
# ──────────────────────────────────────────────────────────────────────────
print("Computing Tier 1 features (temp_at_hour, HDH, CDH)...")

# Sort by (zone, timestamp) — required for rolling windows below
long_trimmed = long_trimmed.sort_values(["zone_name", "timestamp"]).reset_index(drop=True)

# Rename temp_f → temp_at_hour to match feature naming convention
long_trimmed = long_trimmed.rename(columns={"temp_f": "temp_at_hour"})

# Heating and cooling degree hours
long_trimmed["HDH_at_hour"] = np.maximum(HDH_CDH_BASE_F - long_trimmed["temp_at_hour"], 0.0)
long_trimmed["CDH_at_hour"] = np.maximum(long_trimmed["temp_at_hour"] - HDH_CDH_BASE_F, 0.0)

print(f"  temp_at_hour:  range [{long_trimmed['temp_at_hour'].min():.1f}, {long_trimmed['temp_at_hour'].max():.1f}] °F")
print(f"  HDH_at_hour:   range [{long_trimmed['HDH_at_hour'].min():.1f}, {long_trimmed['HDH_at_hour'].max():.1f}] (nonzero: {(long_trimmed['HDH_at_hour'] > 0).sum():,} / {len(long_trimmed):,} rows)")
print(f"  CDH_at_hour:   range [{long_trimmed['CDH_at_hour'].min():.1f}, {long_trimmed['CDH_at_hour'].max():.1f}] (nonzero: {(long_trimmed['CDH_at_hour'] > 0).sum():,} / {len(long_trimmed):,} rows)")

# ──────────────────────────────────────────────────────────────────────────
# Tier 2: trailing mean features (per-zone rolling windows)
# ──────────────────────────────────────────────────────────────────────────
print(f"\nComputing Tier 2 features (trailing 24h and 168h means)...")

# Per-zone rolling mean via groupby + transform.
# min_periods=1 means even the first row gets a value (= temp_at_hour itself).
long_trimmed["temp_trailing_24h_at_fc"] = long_trimmed.groupby("zone_name", observed=True)["temp_at_hour"].transform(
    lambda s: s.rolling(window=TRAILING_24H_HOURS, min_periods=1).mean()
)

long_trimmed["temp_trailing_168h_at_fc"] = long_trimmed.groupby("zone_name", observed=True)["temp_at_hour"].transform(
    lambda s: s.rolling(window=TRAILING_168H_HOURS, min_periods=1).mean()
)

print(f"  temp_trailing_24h_at_fc:  range [{long_trimmed['temp_trailing_24h_at_fc'].min():.1f}, {long_trimmed['temp_trailing_24h_at_fc'].max():.1f}] °F")
print(f"  temp_trailing_168h_at_fc: range [{long_trimmed['temp_trailing_168h_at_fc'].min():.1f}, {long_trimmed['temp_trailing_168h_at_fc'].max():.1f}] °F")

# ──────────────────────────────────────────────────────────────────────────
# Verification
# ──────────────────────────────────────────────────────────────────────────
# All 5 features should have zero NaN at this point
for col in ["temp_at_hour", "HDH_at_hour", "CDH_at_hour",
            "temp_trailing_24h_at_fc", "temp_trailing_168h_at_fc"]:
    n_nan = long_trimmed[col].isna().sum()
    assert n_nan == 0, f"Found {n_nan} NaN values in {col}"
print(f"\n✓ All 5 weather features computed with zero NaN values")

# Spot-check: first 5 rows of COAS for visual inspection
print(f"\nSpot check — first 5 rows of COAS:")
print(long_trimmed.loc[long_trimmed["zone_name"] == "COAS"].head(5).to_string(index=False))

# Per-zone feature summary statistics
print(f"\nFeature summary per zone (mean values across 2022-2025):")
feature_cols = ["temp_at_hour", "HDH_at_hour", "CDH_at_hour",
                "temp_trailing_24h_at_fc", "temp_trailing_168h_at_fc"]
summary = long_trimmed.groupby("zone_name", observed=True)[feature_cols].mean().round(2)
print(summary.to_string())

# Sanity check: trailing means should be smoother than point-in-time temp
# (lower variance per zone after smoothing)
print(f"\nVariance reduction check (std of trailing 168h / std of temp_at_hour):")
for zone in ZONES:
    z = long_trimmed.loc[long_trimmed["zone_name"] == zone]
    ratio = z["temp_trailing_168h_at_fc"].std() / z["temp_at_hour"].std()
    print(f"  {zone}: {ratio:.3f}  (lower = more smoothing, as expected)")

elapsed = time.time() - t0
print(f"\n✓ Feature engineering complete in {elapsed:.1f}s")
print(f"  Final shape: {long_trimmed.shape[0]:,} rows × {long_trimmed.shape[1]} columns")
print(f"  Columns: {list(long_trimmed.columns)}")

Computing Tier 1 features (temp_at_hour, HDH, CDH)...
  temp_at_hour:  range [1.0, 114.1] °F
  HDH_at_hour:   range [0.0, 64.0] (nonzero: 107,606 / 280,480 rows)
  CDH_at_hour:   range [0.0, 49.1] (nonzero: 172,874 / 280,480 rows)

Computing Tier 2 features (trailing 24h and 168h means)...
  temp_trailing_24h_at_fc:  range [8.7, 98.0] °F
  temp_trailing_168h_at_fc: range [25.7, 95.7] °F

✓ All 5 weather features computed with zero NaN values

Spot check — first 5 rows of COAS:
          timestamp  temp_c zone_name  temp_at_hour  HDH_at_hour  CDH_at_hour  temp_trailing_24h_at_fc  temp_trailing_168h_at_fc
2022-01-01 00:00:00    23.0      COAS          73.4          0.0          8.4                     73.4                      73.4
2022-01-01 01:00:00    23.0      COAS          73.4          0.0          8.4                     73.4                      73.4
2022-01-01 02:00:00    23.0      COAS          73.4          0.0          8.4                     73.4                      73.4
20

### Feature engineering — observations

All 5 features computed cleanly with zero NaN values. The shape (280,480 rows × 8 columns) reflects the 4 input columns (timestamp, temp_c, zone_name, temp_at_hour) plus 4 new feature columns (HDH_at_hour, CDH_at_hour, temp_trailing_24h_at_fc, temp_trailing_168h_at_fc).

**Heating vs cooling balance across Texas.** Across 280,480 zone-hour observations, 107,606 hours (38%) have nonzero HDH and 172,874 hours (62%) have nonzero CDH. This 38:62 split matches the literature consensus that Texas load is cooling-dominated overall (Hong & Fan 2016 reports similar ratios for Texas/Florida/Southwest US grids). The model will learn separate gradients for HDH and CDH, capturing the asymmetry that heating-driven and cooling-driven load have different per-degree elasticities.

**Per-zone HDH/CDH means reveal climate differences:**

| Zone | HDH mean | CDH mean | Climate signal |
|---|---|---|---|
| COAS (Houston) | 3.07 | 10.71 | Subtropical, cooling-dominant |
| SOUT (San Antonio) | 3.86 | 10.64 | Subtropical |
| SCEN (Austin) | 4.77 | 9.95 | Subtropical edge |
| EAST (Tyler) | 5.45 | 8.79 | Humid subtropical, more balanced |
| NCEN (DFW) | 5.66 | 9.55 | Humid subtropical |
| WEST (San Angelo) | 6.27 | 9.46 | Semi-arid |
| FWES (Midland) | 7.16 | 8.65 | Desert plateau |
| NOTH (Lubbock) | 8.88 | 7.36 | High plains, only heating-dominant zone |

NOTH is the only zone where HDH mean exceeds CDH mean. Lubbock's 1003m elevation and continental position produce more heating-driven hours per year than the gulf-influenced zones. This is a real signal that LightGBM will likely exploit when learning per-zone bias terms.

**Variance reduction from trailing-168h smoothing** ranged from 0.810 (WEST) to 0.869 (NCEN). All zones fell in the expected 0.75-0.95 band — the 168h mean is informatively smoother than the raw temp but not so flat that it loses seasonal signal. This is the right balance for the trailing-mean feature's purpose: capturing seasonal context without competing with the temp_at_hour feature for point-in-time signal.

**The first hours of each zone's series get rolling means with shorter windows** (min_periods=1 fallback). The COAS spot-check shows temp_trailing_24h_at_fc identical to temp_at_hour for the first 5 hours — correct behavior given the entire night of Jan 1, 2022 was a steady 73.4°F at Houston Hobby. By hour 24 the rolling window is fully populated; by hour 168 the weekly window is fully populated. Edge effects affect only the first 167 hours out of 35,060 per zone (0.5%).

**No surprises in the feature distributions.** All ranges are physically plausible: HDH peaks at 64 (corresponding to the 1°F observation at NOTH), CDH peaks at 49.1 (corresponding to the 114.1°F observation at WEST). Both are within the realistic envelope for Texas weather extremes. The features are ready to write to disk.

In [9]:
"""
Final verification and write the weather features parquet.

Process:
  1. Select only the columns we need for the output (drop temp_c, keep temp_at_hour
     and the 4 engineered features plus the (zone_name, timestamp) join key).
  2. Cast numeric columns to float32 (matches notebook 02's convention for
     non-target features).
  3. Cast zone_name to pandas categorical (matches notebook 02's convention).
  4. Write to data/processed/weather_features/weather_features.parquet.
  5. Read it back and verify schema, dtypes, row count, no NaN, and a spot-check
     of the round-trip.

Runtime: <2 seconds.
"""

t0 = time.time()

# ──────────────────────────────────────────────────────────────────────────
# Select and cast columns for the output
# ──────────────────────────────────────────────────────────────────────────
print("Preparing output DataFrame...")

OUTPUT_COLUMNS = [
    "zone_name",
    "timestamp",
    "temp_at_hour",
    "HDH_at_hour",
    "CDH_at_hour",
    "temp_trailing_24h_at_fc",
    "temp_trailing_168h_at_fc",
]

output_df = long_trimmed[OUTPUT_COLUMNS].copy()

# Cast dtypes
output_df["zone_name"] = output_df["zone_name"].astype("category")
for col in ["temp_at_hour", "HDH_at_hour", "CDH_at_hour",
            "temp_trailing_24h_at_fc", "temp_trailing_168h_at_fc"]:
    output_df[col] = output_df[col].astype("float32")

print(f"  Output shape: {output_df.shape[0]:,} rows × {output_df.shape[1]} columns")
print(f"  Dtypes:")
for col, dtype in output_df.dtypes.items():
    print(f"    {col:<32} {str(dtype)}")

# ──────────────────────────────────────────────────────────────────────────
# Write parquet
# ──────────────────────────────────────────────────────────────────────────
print(f"\nWriting to {OUTPUT_PATH}...")
output_df.to_parquet(OUTPUT_PATH, index=False, compression="zstd")

size_mb = OUTPUT_PATH.stat().st_size / 1024**2
print(f"  File size: {size_mb:.2f} MB")

# ──────────────────────────────────────────────────────────────────────────
# Round-trip verification: read back and check
# ──────────────────────────────────────────────────────────────────────────
print(f"\nRound-trip verification...")
read_back = pd.read_parquet(OUTPUT_PATH)

# Schema check
assert list(read_back.columns) == OUTPUT_COLUMNS, (
    f"Schema mismatch. Expected {OUTPUT_COLUMNS}, got {list(read_back.columns)}"
)
print(f"  Schema: ✓ all 7 columns in correct order")

# Row count
assert len(read_back) == len(output_df), (
    f"Row count mismatch: wrote {len(output_df):,}, read {len(read_back):,}"
)
print(f"  Row count: ✓ {len(read_back):,} rows")

# Zero NaN
for col in OUTPUT_COLUMNS[2:]:  # skip zone_name and timestamp
    n_nan = read_back[col].isna().sum()
    assert n_nan == 0, f"Found {n_nan} NaN in {col} after read-back"
print(f"  NaN check: ✓ 0 NaN values in any numeric column")

# Zone coverage
read_zones = sorted(read_back["zone_name"].unique())
assert read_zones == ZONES, f"Zone mismatch on read-back: {read_zones}"
print(f"  Zone coverage: ✓ all 8 zones ({read_zones})")

# Per-zone row counts
print(f"  Per-zone row counts:")
for zone in ZONES:
    n = (read_back["zone_name"] == zone).sum()
    print(f"    {zone}: {n:>6,}")

# Value spot-check: first row of COAS should match what we computed
first_coas = read_back[read_back["zone_name"] == "COAS"].iloc[0]
print(f"\n  Spot check — first COAS row from parquet:")
for col in OUTPUT_COLUMNS:
    print(f"    {col:<32} {first_coas[col]}")

elapsed = time.time() - t0
print(f"\n✓ Weather features written and verified in {elapsed:.1f}s")
print(f"  Output: {OUTPUT_PATH.resolve()}")
print(f"  Size:   {size_mb:.2f} MB")
print(f"  Schema: 7 columns ({len(OUTPUT_COLUMNS)} total)")
print(f"  Rows:   {len(read_back):,}")

Preparing output DataFrame...
  Output shape: 280,480 rows × 7 columns
  Dtypes:
    zone_name                        category
    timestamp                        datetime64[us]
    temp_at_hour                     float32
    HDH_at_hour                      float32
    CDH_at_hour                      float32
    temp_trailing_24h_at_fc          float32
    temp_trailing_168h_at_fc         float32

Writing to ../data/processed/weather_features/weather_features.parquet...
  File size: 2.39 MB

Round-trip verification...
  Schema: ✓ all 7 columns in correct order
  Row count: ✓ 280,480 rows
  NaN check: ✓ 0 NaN values in any numeric column
  Zone coverage: ✓ all 8 zones (['COAS', 'EAST', 'FWES', 'NCEN', 'NOTH', 'SCEN', 'SOUT', 'WEST'])
  Per-zone row counts:
    COAS: 35,060
    EAST: 35,060
    FWES: 35,060
    NCEN: 35,060
    NOTH: 35,060
    SCEN: 35,060
    SOUT: 35,060
    WEST: 35,060

  Spot check — first COAS row from parquet:
    zone_name                        COAS
    tim

### Output file written and verified

The weather features parquet is now persisted to disk and verified via round-trip read:

| Property | Value |
|---|---|
| Path | `data/processed/weather_features/weather_features.parquet` |
| Size | 2.39 MB |
| Rows | 280,480 |
| Columns | 7 (zone_name, timestamp, temp_at_hour, HDH_at_hour, CDH_at_hour, temp_trailing_24h_at_fc, temp_trailing_168h_at_fc) |
| Dtypes | zone_name=category, timestamp=datetime64[us], features=float32 |
| Per-zone row count | 35,060 (8 zones × 35,060 hours = 280,480 total) |

The file is small enough to commit to git without size concerns (well under GitHub's 100 MB per-file limit). The downstream notebooks 04b and 05b will read this file alongside the notebook 02 feature parquets and merge them via `(zone_name, timestamp)`.

**Notebook 02w is complete.** The weather feature foundation is in place. Three implementation choices are worth restating for documentation:

1. **Concurrent weather assumption (Option 5A)**: the `_at_hour` features represent observed temperature at the prediction hour. In a production system this would be replaced by a high-quality short-horizon weather forecast (~1°F MAE for next-day). We treat observed temperature as a proxy for forecasted temperature, consistent with Hong & Fan 2016 and the broader load forecasting literature.

2. **Trailing means computed relative to timestamp, not task-specific forecast time**: the 24h and 168h means are simple rolling windows over the temperature time series. Notebook 02's pd-based trailing means are task-aware (different lookback windows per task), but for weather we use a single canonical version. The small admissibility cut (using temperature within the trailing window even if it falls within the forecast horizon) is documented as a limitation in the report.

3. **DST fall-back resolution via `keep='first'`**: the four ambiguous (zone, 01:00) hours per year are resolved by keeping the pre-transition observation. This drops 4 hours per zone per year (32 hours total across 4 years × 8 zones), giving 35,060 rows per zone instead of 35,064. The downstream join with notebook 02's feature parquets will handle these dropped hours automatically via the left-join semantics — rows in the feature matrix that don't have a matching weather row will simply have NaN for the weather features, which LightGBM handles natively.

Next: commit notebook 02w + weather_features.parquet to git, then proceed to notebook 04b (weather-augmented zone-direct LightGBM).